# Project Title: Examining the Relationship Between IBM News Coverage and Market Behavior


by Sarang Pinakin Kadakia

## Project objective:







The goal of this project is to understand whether IBM-related news aligns with movements in IBM’s stock performance. I analyze daily stock prices, trading volume, volatility, and news sentiment to see if changes in media activity appear alongside changes in market behavior.

## Data Overview

In this project, I worked with two different real-world datasets that I pulled using public APIs.

### 1. IBM Stock Market Data (Alpha Vantage API)

- **Source:** Alpha Vantage — 'TIME_SERIES_DAILY' endpoint  
- **Period Covered:** July 18, 2025 to December 8, 2025  
- **Number of Observations:** 100 daily stock records  
- **Includes:** Open, high, low, close prices, and trading volume  

### 2. IBM News Dataset (Mediastack API + Google NLU Sentiment)

- **Source:** Mediastack News API (with pagination)  
- **Period Covered:** September 1, 2025 to December 8, 2025  
- **Number of Articles Collected:** 824 news articles after duplicate removal  
- **Includes:** Headlines, descriptions, publication dates, and sentiment scores computed using Google Cloud Natural Language  

### Final Dataset

After cleaning and aligning dates from both sources, the datasets were merged into **final_df**, which contains:

- **100 fully matched daily records**  
- Each row includes key metrics such as stock metrics, news volume, and average sentiment for that date.


### Content
This notebook includes all required components for Phases 1–4 of the final project:

1. d1: Library Imports

2. d2: Data Pre-Processing (stock data + Mediastack news + Google NLU sentiment)

3. d3: Data Analysis (five original research questions with interactive Bokeh visualizations)

4. d4: Summary of Findings

5. d5: Future Research

## D1. Library imports

In [87]:
import os
import requests
from IPython.display import display
from datetime import datetime
import time
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from bokeh.palettes import Category10
from bokeh.transform import factor_cmap
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import HoverTool, ColumnDataSource, LinearAxis, Range1d
from google.cloud import language_v1
from bokeh.io import output_notebook

In [88]:
!pip install google-cloud-language

##D2. Data pre-processing


### Why I chose (Alpha Vantage & Mediastack) APIs
I wanted to look at something real and easy to relate to. Stock prices often react to news, so I paired two datasets:

1. IBM’s daily stock prices: Alpha Vantage API

2. IBM-related news articles: Mediastack API

Putting these together helps me explore one main idea:

**Do news events show any connection with IBM’s stock movements?**

# A: Load IBM Stock Data (Alpha Vantage)

Here I download daily IBM stock data from the Alpha Vantage API using their api key.
I convert the JSON into a DataFrame, make sure the numeric columns are actually numbers, and convert the date column so I can merge it with the news dataset later.

In [89]:
alpha_url = "https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=IBM&apikey=HFO2S05VUETPUPC5"
alpha_data = requests.get(alpha_url).json()
print(alpha_data)

{'Meta Data': {'1. Information': 'Daily Prices (open, high, low, close) and Volumes', '2. Symbol': 'IBM', '3. Last Refreshed': '2025-12-08', '4. Output Size': 'Compact', '5. Time Zone': 'US/Eastern'}, 'Time Series (Daily)': {'2025-12-08': {'1. open': '309.6200', '2. high': '315.3454', '3. low': '307.9500', '4. close': '309.1800', '5. volume': '3615794'}, '2025-12-05': {'1. open': '308.5900', '2. high': '311.8300', '3. low': '307.1800', '4. close': '307.9400', '5. volume': '2344667'}, '2025-12-04': {'1. open': '302.8750', '2. high': '309.6100', '3. low': '302.5400', '4. close': '307.9900', '5. volume': '2962463'}, '2025-12-03': {'1. open': '302.8800', '2. high': '303.9700', '3. low': '298.9050', '4. close': '302.6200', '5. volume': '3953390'}, '2025-12-02': {'1. open': '307.0000', '2. high': '310.4675', '3. low': '301.5700', '4. close': '301.7800', '5. volume': '4261100'}, '2025-12-01': {'1. open': '306.5050', '2. high': '307.1200', '3. low': '302.8000', '4. close': '305.6700', '5. volu

In [90]:
# Extracting the time series part
ts = alpha_data["Time Series (Daily)"]
print(ts)

{'2025-12-08': {'1. open': '309.6200', '2. high': '315.3454', '3. low': '307.9500', '4. close': '309.1800', '5. volume': '3615794'}, '2025-12-05': {'1. open': '308.5900', '2. high': '311.8300', '3. low': '307.1800', '4. close': '307.9400', '5. volume': '2344667'}, '2025-12-04': {'1. open': '302.8750', '2. high': '309.6100', '3. low': '302.5400', '4. close': '307.9900', '5. volume': '2962463'}, '2025-12-03': {'1. open': '302.8800', '2. high': '303.9700', '3. low': '298.9050', '4. close': '302.6200', '5. volume': '3953390'}, '2025-12-02': {'1. open': '307.0000', '2. high': '310.4675', '3. low': '301.5700', '4. close': '301.7800', '5. volume': '4261100'}, '2025-12-01': {'1. open': '306.5050', '2. high': '307.1200', '3. low': '302.8000', '4. close': '305.6700', '5. volume': '3166555'}, '2025-11-28': {'1. open': '304.0600', '2. high': '309.1800', '3. low': '303.6000', '4. close': '308.5800', '5. volume': '1689031'}, '2025-11-26': {'1. open': '305.1800', '2. high': '306.6000', '3. low': '301

In [91]:
stock_df = pd.DataFrame.from_dict(ts, orient = "index").reset_index()
print(stock_df)

         index   1. open   2. high    3. low  4. close 5. volume
0   2025-12-08  309.6200  315.3454  307.9500  309.1800   3615794
1   2025-12-05  308.5900  311.8300  307.1800  307.9400   2344667
2   2025-12-04  302.8750  309.6100  302.5400  307.9900   2962463
3   2025-12-03  302.8800  303.9700  298.9050  302.6200   3953390
4   2025-12-02  307.0000  310.4675  301.5700  301.7800   4261100
..         ...       ...       ...       ...       ...       ...
95  2025-07-24  261.2500  262.0486  252.7500  260.5100  22647720
96  2025-07-23  284.3000  288.0800  281.4400  282.0100   8105906
97  2025-07-22  284.7400  284.8800  281.2500  281.9600   4824219
98  2025-07-21  286.2900  287.7300  284.3800  284.7100   3051791
99  2025-07-18  283.3800  287.1600  282.2200  285.8700   4478165

[100 rows x 6 columns]


In [92]:
stock_df.rename(columns={"index": "date"}, inplace=True)
print(stock_df)

          date   1. open   2. high    3. low  4. close 5. volume
0   2025-12-08  309.6200  315.3454  307.9500  309.1800   3615794
1   2025-12-05  308.5900  311.8300  307.1800  307.9400   2344667
2   2025-12-04  302.8750  309.6100  302.5400  307.9900   2962463
3   2025-12-03  302.8800  303.9700  298.9050  302.6200   3953390
4   2025-12-02  307.0000  310.4675  301.5700  301.7800   4261100
..         ...       ...       ...       ...       ...       ...
95  2025-07-24  261.2500  262.0486  252.7500  260.5100  22647720
96  2025-07-23  284.3000  288.0800  281.4400  282.0100   8105906
97  2025-07-22  284.7400  284.8800  281.2500  281.9600   4824219
98  2025-07-21  286.2900  287.7300  284.3800  284.7100   3051791
99  2025-07-18  283.3800  287.1600  282.2200  285.8700   4478165

[100 rows x 6 columns]


In [93]:
# Converting all numeric columns to floats
for col in stock_df.columns:
  if col != "date":
    stock_df[col] = stock_df[col].astype(float)

In [94]:
# Converting date to datetime format
stock_df["date"] = pd.to_datetime(stock_df["date"])

In [95]:
stock_df.head()

,date,1. open,2. high,3. low,4. close,5. volume
0,2025-12-08,309.620,315.3454,307.950,309.18,3615794.0
1,2025-12-05,308.590,311.8300,307.180,307.94,2344667.0
2,2025-12-04,302.875,309.6100,302.540,307.99,2962463.0
3,2025-12-03,302.880,303.9700,298.905,302.62,3953390.0
4,2025-12-02,307.000,310.4675,301.570,301.78,4261100.0


In [96]:
# Adding useful financial metrics to stock data
"""
These new columns help me measure:

a.'daily_change': how much the stock moved that day

b. 'daily_return': the day-to-day percentage gain or loss

c.'percent_change': percent return

d. 'volatility': difference between high and low prices

e. 'volatility_pct': how large the volatility was relative to the opening price
"""
stock_df["daily_change"] = stock_df["4. close"] - stock_df["1. open"]
stock_df["daily_return"] = stock_df["4. close"].pct_change() * 100
stock_df["percent_change"] = (stock_df["daily_change"] / stock_df["1. open"])*100
stock_df["volatility"] = stock_df["2. high"] - stock_df["3. low"]
stock_df["volatility_pct"] = (stock_df["volatility"] / stock_df["1. open"])*100

In [97]:
stock_df.head()

,date,1. open,2. high,3. low,4. close,5. volume,daily_change,daily_return,percent_change,volatility,volatility_pct
0,2025-12-08,309.620,315.3454,307.950,309.18,3615794.0,-0.440,NaN,-0.142110,7.3954,2.388541
1,2025-12-05,308.590,311.8300,307.180,307.94,2344667.0,-0.650,-0.401061,-0.210635,4.6500,1.506854
2,2025-12-04,302.875,309.6100,302.540,307.99,2962463.0,5.115,0.016237,1.688816,7.0700,2.334296
3,2025-12-03,302.880,303.9700,298.905,302.62,3953390.0,-0.260,-1.743563,-0.085843,5.0650,1.672279
4,2025-12-02,307.000,310.4675,301.570,301.78,4261100.0,-5.220,-0.277576,-1.700326,8.8975,2.898208


# B: Load IBM-related News Data (Mediastack)

Here I pull IBM-related news articles. Each row represents a single article.
I extract the date from the timestamp because I want to match each article to daily stock movements.

In [98]:
min_date = stock_df['date'].min().strftime('%Y-%m-%d')
max_date = min(stock_df['date'].max(), datetime.today()).strftime('%Y-%m-%d')

news_url = "https://api.mediastack.com/v1/news?access_key=3143dd96a40d4d197bc9d88c3e75a12b&keywords=IBM&countries=us"

# Appending date_from and date_to parameters (to match it with stock_data)
news_url_updated = f"{news_url}&date_from={min_date}&date_to={max_date}"

print(f"Fetching news data from: {news_url_updated}")

# Applying pagination
"""
I used pagination because APIs often limit the number of results we get per request, even when specifying a date range.
This method helped me repeatedly ask for more data until I collected everything available.
"""
all_news_data = []
limit = 100
offset = 0
total_articles = 1

while offset < total_articles:
    current_news_url = f"{news_url_updated}&limit={limit}&offset={offset}"
    # print(f"Fetching page with offset: {offset}")
    news_json_page = requests.get(current_news_url).json()

    if not news_json_page or 'data' not in news_json_page or not news_json_page['data']:
        print(f"No more data or an error occurred at offset {offset}.")
        break

    if offset == 0:
        total_articles = news_json_page['pagination']['total']
        print(f"Total articles found by API for this range: {total_articles}")

    all_news_data.extend(news_json_page['data'])

    # if the current page returned fewer articles than the limit or if we have collected all available articles (to prevent infinite loops if total is wrong)
    if news_json_page['pagination']['count'] < limit or (offset + limit) >= total_articles:
        break

    offset += limit

news_data = all_news_data
print(f"Successfully fetched {len(news_data)} news articles using pagination.")

print(news_data)

Fetching news data from: https://api.mediastack.com/v1/news?access_key=3143dd96a40d4d197bc9d88c3e75a12b&keywords=IBM&countries=us&date_from=2025-07-18&date_to=2025-12-08
Total articles found by API for this range: 827
Successfully fetched 827 news articles using pagination.
[{'author': 'Aparajita Chatterjee', 'title': 'IBM acquisition announcement spurs Confluent stock to soar', 'description': 'IBMshares rose 0.5% on Dec. 8, adding to its 41% year-to-date stock gain, after the consulting provider announced its plans to acquire the real-time data streaming platform Confluent.&nbsp; More importantly, though, Confluent’s stock price soared 29% following the news, not its highest price at ...', 'url': 'https://www.thestreet.com/investing/stocks/ibm-acquisition-announcement-spurs-confluent-stock-to-soar', 'source': 'thestreet', 'image': 'https://www.thestreet.com/.image/c_fit%2Ch_800%2Cw_1200/NDA6MDAwMDAwMDAyNzkzMjIw/ibmlogoonaglassbuildingitcompanyheadquartersisrael.jpg', 'category': 'gene

In [99]:
news_df = pd.json_normalize(news_data)

In [100]:
# Converting published_at to date only
news_df["date"] = pd.to_datetime(news_df["published_at"]).dt.date
news_df["date"] = pd.to_datetime(news_df["date"])

In [101]:
news_df.head()

,author,title,description,url,source,image,category,language,country,published_at,date
0,Aparajita Chatterjee,IBM acquisition announcement spurs Confluent s...,"IBMshares rose 0.5% on Dec. 8, adding to its 4...",https://www.thestreet.com/investing/stocks/ibm...,thestreet,https://www.thestreet.com/.image/c_fit%2Ch_800...,general,en,us,2025-12-08T21:06:18+00:00,2025-12-08
1,None,IBM to buy Confluent to grow AI data processing,IBM announced its multi-billion-dollar buyout ...,https://www.upi.com/Top_News/US/2025/12/08/IBM...,upiasia,https://cdnph.upi.com/ph/st/th/1241765206609/2...,general,en,us,2025-12-08T16:11:41+00:00,2025-12-08
2,Ram Iyer,IBM to acquire Confluent for $11B as it seeks ...,IBM is buying data infrastructure company Conf...,https://techcrunch.com/2025/12/08/ibm-to-acqui...,TechCrunch,None,technology,en,us,2025-12-08T14:56:10+00:00,2025-12-08
3,None,IBM buys data streaming platform Confluent in ...,IBM has announced it's acquiring data streamin...,https://www.marketbeat.com/articles/ibm-buys-d...,lulegacy,None,general,en,us,2025-12-08T14:07:32+00:00,2025-12-08
4,None,IBM acquiring Confluent in $11 billion all-cas...,IBM will pay $31 per share in cash for all of ...,https://www.cnbc.com/2025/12/08/ibm-confluent-...,CNBC,None,general,en,us,2025-12-08T13:05:11+00:00,2025-12-08


In [102]:
# Renaming columns
column_name_mapping = {
    '1. open': 'open',
    '2. high': 'high',
    '3. low': 'low',
    '4. close': 'close',
    '5. volume': 'volume'
}

stock_df.rename(columns=column_name_mapping, inplace=True)

print("Columns renamed in stock_df:")
print(stock_df.columns)

Columns renamed in stock_df:
Index(['date', 'open', 'high', 'low', 'close', 'volume', 'daily_change',
       'daily_return', 'percent_change', 'volatility', 'volatility_pct'],
      dtype='object')


In [103]:
# Adding sentiment score using Google NLU
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [104]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/drive/MyDrive/dwd-finalproject-7e76a3ed6b98.json"
!gcloud auth activate-service-account --key-file "/content/drive/MyDrive/dwd-finalproject-7e76a3ed6b98.json"

Activated service account credentials for: [nlu-access@dwd-finalproject.iam.gserviceaccount.com]


In [105]:
client = language_v1.LanguageServiceClient()

In [106]:
# Adding sentiment score using Google's NLU API
"""
Here, I create a new column called sentiment by analyzing the tone of each news article. Sentiment ranges from -1 (negative) to +1 (positive).
This helps me check if negative news correlates with drops in IBM stock price.
"""
def get_sentiment_google(text):
    if not text or text.strip() == "":
        return 0.0

    document = language_v1.Document(
        content=text,
        type_=language_v1.Document.Type.PLAIN_TEXT,
        language="en"
    )

    response = client.analyze_sentiment(request={"document": document})
    return response.document_sentiment.score

In [107]:
# Now I will compute sentiment from title + description combined
news_df["combined_text"] = news_df["title"].fillna("") + " " + news_df["description"].fillna("")
news_df["sentiment"] = news_df["combined_text"].apply(get_sentiment_google)

In [108]:
news_df[["date", "sentiment"]].head()

,date,sentiment
0,2025-12-08,0.0
1,2025-12-08,-0.4
2,2025-12-08,0.1
3,2025-12-08,-0.2
4,2025-12-08,0.1


In [109]:
# Grouping news by Day (to merge woth stock data)
"""
Since stock data is daily, I convert the news dataset into a daily-level summary.
For each date, I calculate:
a. the number of articles
b. their average sentiment
This allows me to align the news information with the stock information.
"""
daily_news = news_df.groupby("date").agg(
    news_count=("sentiment", "count"),
    avg_sentiment=("sentiment", "mean")
).reset_index()

daily_news.head()

,date,news_count,avg_sentiment
0,2025-09-01,3,-0.033333
1,2025-09-02,5,0.020000
2,2025-09-03,3,0.033333
3,2025-09-04,4,0.000000
4,2025-09-05,6,0.066667


In [110]:
# Converting news_count to Integer
daily_news["news_count"] = daily_news["news_count"].astype(int)

print("news_count column converted to integer type.")
daily_news.info()
# daily_news.head()

news_count column converted to integer type.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           99 non-null     datetime64[ns]
 1   news_count     99 non-null     int64         
 2   avg_sentiment  99 non-null     float64       
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 2.4 KB


In [111]:
# Checking for duplicate News Articles
print(f"Number of news articles before removing duplicates: {len(news_df)}")

# Removing duplicate rows based on 'title' and 'description'
news_df.drop_duplicates(subset=['title', 'description'], inplace=True)

print(f"Number of news articles after removing duplicates: {len(news_df)}")

Number of news articles before removing duplicates: 827
Number of news articles after removing duplicates: 824


In [112]:
# Finally, merging stock + news datasets
"""
Finally, I merge both datasets on the date column.
If a date has no news, I set the counts and sentiment scores to 0.
This gives me one complete dataset that captures both financial behavior and media activity for IBM.
"""
merged_df = pd.merge(stock_df, daily_news, on = "date", how = "left")

# Filling missing news for days with no articles
merged_df["news_count"] = merged_df["news_count"].fillna(0)
merged_df["avg_sentiment"] = merged_df["avg_sentiment"].fillna(0)

merged_df.head()

,date,open,high,low,close,volume,daily_change,daily_return,percent_change,volatility,volatility_pct,news_count,avg_sentiment
0,2025-12-08,309.620,315.3454,307.950,309.18,3615794.0,-0.440,NaN,-0.142110,7.3954,2.388541,16.0,-0.00625
1,2025-12-05,308.590,311.8300,307.180,307.94,2344667.0,-0.650,-0.401061,-0.210635,4.6500,1.506854,6.0,-0.05000
2,2025-12-04,302.875,309.6100,302.540,307.99,2962463.0,5.115,0.016237,1.688816,7.0700,2.334296,10.0,0.06000
3,2025-12-03,302.880,303.9700,298.905,302.62,3953390.0,-0.260,-1.743563,-0.085843,5.0650,1.672279,5.0,-0.08000
4,2025-12-02,307.000,310.4675,301.570,301.78,4261100.0,-5.220,-0.277576,-1.700326,8.8975,2.898208,4.0,0.05000


In [113]:
final_df = merged_df.sort_values("date", ascending= False)
final_df.head()

,date,open,high,low,close,volume,daily_change,daily_return,percent_change,volatility,volatility_pct,news_count,avg_sentiment
0,2025-12-08,309.620,315.3454,307.950,309.18,3615794.0,-0.440,NaN,-0.142110,7.3954,2.388541,16.0,-0.00625
1,2025-12-05,308.590,311.8300,307.180,307.94,2344667.0,-0.650,-0.401061,-0.210635,4.6500,1.506854,6.0,-0.05000
2,2025-12-04,302.875,309.6100,302.540,307.99,2962463.0,5.115,0.016237,1.688816,7.0700,2.334296,10.0,0.06000
3,2025-12-03,302.880,303.9700,298.905,302.62,3953390.0,-0.260,-1.743563,-0.085843,5.0650,1.672279,5.0,-0.08000
4,2025-12-02,307.000,310.4675,301.570,301.78,4261100.0,-5.220,-0.277576,-1.700326,8.8975,2.898208,4.0,0.05000


In [114]:
# Just validating date range overlap to ensure consistent and expected time period for analysis
print(f"Stock data date range: {stock_df['date'].min()} to {stock_df['date'].max()}")
print(f"News data date range: {daily_news['date'].min()} to {daily_news['date'].max()}")
print(f"Final data date range: {final_df['date'].min()} to {final_df['date'].max()}")

Stock data date range: 2025-07-18 00:00:00 to 2025-12-08 00:00:00
News data date range: 2025-09-01 00:00:00 to 2025-12-08 00:00:00
Final data date range: 2025-07-18 00:00:00 to 2025-12-08 00:00:00


In [115]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   date            100 non-null    datetime64[ns]
 1   open            100 non-null    float64       
 2   high            100 non-null    float64       
 3   low             100 non-null    float64       
 4   close           100 non-null    float64       
 5   volume          100 non-null    float64       
 6   daily_change    100 non-null    float64       
 7   daily_return    99 non-null     float64       
 8   percent_change  100 non-null    float64       
 9   volatility      100 non-null    float64       
 10  volatility_pct  100 non-null    float64       
 11  news_count      100 non-null    float64       
 12  avg_sentiment   100 non-null    float64       
dtypes: datetime64[ns](1), float64(12)
memory usage: 10.3 KB


In [116]:
# Exporting the final merged dataset to a CSV file
final_df.to_csv("ibm_final_dataset.csv", index=False)

print("Final dataset exported as: ibm_final_dataset.csv")

Final dataset exported as: ibm_final_dataset.csv


## D3. Data analysis



### What I will do in this notebook
In this phase, I move from preparing the data to actually analyzing it. I’ll explore five questions that connect IBM’s stock behavior with IBM-related news activity. To do this, I’ll use a mix of simple analytical tools such as Bokeh visualizations, correlations, sentiment comparisons, volatility patterns, and a basic clustering method.

Even though I will run all five analyses in the notebook, I will highlight only two-three of the most important findings in the final(live) presentation, as required.

Each question will use a different analytical technique such as:
- interactive time-series visualizations (Bokeh)
- correlation analysis
- sentiment-based comparisons
- volatility vs. news volume patterns
- simple classification or clustering



## Multiseries Line Chart of IBM Stock Price Trends Over Time
### Question 1. How did IBM’s Open, Close, High, and Low prices move over time?



#### Answer
This is my starting point for the analysis because it helps me understand the overall behavior of IBM stock independent of the news dataset. Before studying relationships or correlations, I first want a clear picture of how the stock itself is moving day-to-day.

**Analysis method**

For this question, I will use a multiseries line chart, implemented using Bokeh so the visualization becomes interactive, zoomable, easy to explore, clear and visually appealing

The plot will include:

1. Open price

2. Close price

3. High price

4. Low price

All plotted on the same figure with a legend for toggling series visibility.

In [117]:
output_notebook()

# Preparing the data for Bokeh
source = ColumnDataSource(data={
    "date": final_df["date"],
    "open": final_df["open"],
    "close": final_df["close"],
    "high": final_df["high"],
    "low": final_df["low"],
})

# Creating the figure
p = figure(
    x_axis_type="datetime",
    width=900,
    height=450,
    title="IBM Stock Prices Over Time (Open, Close, High, Low)",
    toolbar_location="above"
)

# Adding lines for each series
p.line("date", "open", source=source, line_width=2, legend_label="Open Price")
p.line("date", "close", source=source, line_width=2, legend_label="Close Price")
p.line("date", "high", source=source, line_width=2, legend_label="High Price")
p.line("date", "low", source=source, line_width=2, legend_label="Low Price")

# Add hover tool for interactive inspection
hover = HoverTool(
    tooltips=[
        ("Date", "@date{%F}"),
        ("Open", "@open"),
        ("Close", "@close"),
        ("High", "@high"),
        ("Low", "@low"),
    ],
    formatters={"@date": "datetime"},
    mode="vline"
)
p.add_tools(hover)

# Some Styling
p.legend.location = "top_left"
p.legend.click_policy = "hide"  # Allows toggling series on/off
p.xaxis.axis_label = "Date"
p.yaxis.axis_label = "Price (USD)"

# Show the plot
show(p)

## News Sentiment vs IBM Daily Stock Returns
### Question2. How do IBM’s daily stock returns move compared to the sentiment of IBM-related news on the same days?



#### Answer
I want to see if the tone of IBM news (more positive or negative) lines up with how the stock actually moves day-to-day. Plotting both series over time helps me visually check if they sometimes move in the same direction.

**Analysis method**

To answer this question, I will:
- Compute daily return (%) from the closing price.  
- Use the average news sentiment per day.  
- Create a Bokeh line chart with both series over time (two y-axes: one for return, one for sentiment).

I will consider the following attributes:
- date  
- close -> used to calculate 'daily_return'
- avg_sentiment

This should return an interactive Bokeh plot showing daily return and sentiment on the same timeline so I can visually inspect whether they rise or fall together in certain periods.

In [119]:
output_notebook()

q3_df = merged_df.copy().sort_values("date")

# Computing daily return (%) from close price
q3_df["daily_return"] = final_df["daily_return"]

# Dropping first row with NaN daily_return
q3_df = q3_df.dropna(subset=["daily_return"])

# Preparing data for Bokeh
source = ColumnDataSource(data={
    "date": q3_df["date"],
    "daily_return": q3_df["daily_return"],
    "avg_sentiment": q3_df["avg_sentiment"],
})

# Creating figure
p3 = figure(
    x_axis_type="datetime",
    width=900,
    height=450,
    title="IBM Daily Stock Return vs News Sentiment Over Time",
    toolbar_location="above"
)

# First line: daily return (%) on left y-axis
p3.line(
    "date",
    "daily_return",
    source=source,
    line_width=2,
    color="navy",
    legend_label="Daily Return (%)"
)

p3.yaxis.axis_label = "Daily Return (%)"

# Second y-axis for sentiment
p3.extra_y_ranges = {"sentiment_axis": Range1d(
    start=q3_df["avg_sentiment"].min() * 1.1,
    end=q3_df["avg_sentiment"].max() * 1.1
)}
p3.add_layout(LinearAxis(y_range_name="sentiment_axis", axis_label="Average Sentiment"), 'right')

# Second line: sentiment on right y-axis
p3.line(
    "date",
    "avg_sentiment",
    source=source,
    line_width=2,
    color="orange",
    legend_label="Average Sentiment",
    y_range_name="sentiment_axis"
)

# Adding hover tool
hover = HoverTool(
    tooltips=[
        ("Date", "@date{%F}"),
        ("Daily Return (%)", "@daily_return{0.2f}"),
        ("Avg Sentiment", "@avg_sentiment{0.2f}")
    ],
    formatters={"@date": "datetime"},
    mode="vline"
)
p3.add_tools(hover)

p3.legend.location = "top_left"
p3.legend.click_policy = "hide"
p3.xaxis.axis_label = "Date"

show(p3)

## Relationship Between News Volume and IBM Trading Volume
### Question3. Do days with higher numbers of IBM-related news articles correspond to higher trading volume?



#### Answer
After understanding the general stock trends, I now want to explore whether media activity aligns with how heavily IBM stock is traded.
If a lot of news comes out on a particular day, it may influence trader behavior. Even if it doesn’t always change the stock price, it can increase activity, which is reflected in trading volume.

This question helps me explore whether news volume and market activity move together in any meaningful way.

**Analysis method**

To answer this question, I will:

1. merge daily article counts with daily trading volume

2. create a scatterplot using Bokeh

3. use a trend line (via simple linear regression) to show the overall relationship

This will help me see whether higher news volume tends to be associated with more trading activity.

In [118]:
output_notebook()

# Firstly I will prepare the data
q2_df = final_df.copy()

# Ensuring numeric types
q2_df["news_count"] = q2_df["news_count"].fillna(0)
q2_df["volume"] = pd.to_numeric(q2_df["volume"], errors='coerce')

# Simple linear regression for trend line
x = q2_df["news_count"].values
y = q2_df["volume"].values

m, b = np.polyfit(x, y, 1)
trend_y = m * x + b

# Creating scatter plot
p2 = figure(
    width=900,
    height=450,
    title="News Volume vs IBM Trading Volume",
    x_axis_label="Number of IBM News Articles",
    y_axis_label="IBM Trading Volume",
    tools="pan,wheel_zoom,box_zoom,reset,hover,save"
)

# Defining scatter points
p2.circle(
    q2_df["news_count"],
    q2_df["volume"],
    size=8,
    alpha=0.6,
    legend_label="Daily Data Points"
)

# Defining trend line
p2.line(
    q2_df["news_count"],
    trend_y,
    line_width=2,
    color="red",
    legend_label="Trend Line"
)

# Adding hover tool formatting
hover = p2.select_one(HoverTool)
hover.tooltips = [
    ("News Articles", "@x"),
    ("Volume", "@y")
]

p2.legend.location = "top_left"

show(p2)

## News Volume vs IBM Stock Volatility

### Question4. Are days with more IBM-related news articles also days when the stock is more volatile?



#### Answer
Now, I want to see whether heavy media coverage around IBM tends to coincide with “busy” days in the market, where the stock price moves more within the same day. Volatility here is a simple way to capture how much the price swings between the high and low of the day.

**Analysis method**  

- Compute a basic volatility measure using the daily high and low prices.  
- Use the total number of IBM-related articles (news_count) per day.  
- Create a Bokeh scatter plot of news volume vs volatility to see whether higher news counts tend to line up with higher daily volatility.

My variables of interest here are:
- high, low, close → to compute volatility  
- news_count -> number of news articles about IBM per day  

Finally, I'll plot a Bokeh scatter chart that lets me visually inspect whether days with more IBM news tend to also be days with larger price swings.

In [120]:
output_notebook()

q4_df = final_df.copy()

# Computing volatility
q4_df["volatility_pct"] = final_df["volatility_pct"]

# Dropping rows with any problematic values
q4_df = q4_df.dropna(subset=["volatility_pct"])

source_q4 = ColumnDataSource(data={
    "news_count": q4_df["news_count"],
    "volatility_pct": q4_df["volatility_pct"],
    "date": q4_df["date"],
})

# Creating Bokeh scatter plot
p4 = figure(
    width=900,
    height=450,
    title="IBM News Volume vs Daily Stock Volatility",
    x_axis_label="Number of IBM News Articles",
    y_axis_label="Volatility (%)",
    toolbar_location="above",
    tools="pan,wheel_zoom,box_zoom,reset,hover,save"
)

# Defining scatter points
p4.circle(
    "news_count",
    "volatility_pct",
    source=source_q4,
    size=8,
    alpha=0.7,
    legend_label="Daily Points"
)

# Adding hover tool
hover_q4 = p4.select_one(HoverTool)
hover_q4.tooltips = [
    ("Date", "@date{%F}"),
    ("Articles", "@news_count"),
    ("Volatility (%)", "@volatility_pct{0.2f}")
]
hover_q4.formatters = {"@date": "datetime"}

p4.legend.location = "top_left"

show(p4)

## Grouping IBM Trading Days into Simple Behavioral Clusters

### Question5. Can we group IBM trading days into simple clusters based on news sentiment, news volume, trading volume, returns, and volatility?



#### Answer
Instead of looking at each metric separately, I want to see if there are “types” of days for IBM’s stock. For example: calm days, high-news volatile days, or strong positive-move days. Clustering lets me summarize many features into a few understandable groups.

**Analysis method**  
I will:
- Use a small set of features: daily return, volatility, trading volume, news article count, and average sentiment.  
- Standardize these features so they are on a similar scale.  
- Apply a simple KMeans clustering model (3 clusters).  
- Visualize the clusters on a 2D scatter plot using Bokeh.

My variables of interest here are:
- daily_return   
- volatility_pct
- volume
- news_count  
- avg_sentiment  

Finally, a Bokeh scatter plot where points are colored by cluster, helping me interpret different “regimes” of IBM’s trading days (for example: low-news calm days vs high-news volatile days).

In [121]:
output_notebook()

q5_df = final_df.copy().sort_values("date")

# Computing daily return (%) from close price
q5_df["daily_return"] = final_df["daily_return"]

# volatility_pct
q5_df["volatility_pct"] = final_df["volatility_pct"]

# Dropping first row with NaN daily_return
q5_df = q5_df.dropna(subset=["daily_return", "volatility_pct", "volume"])

# Now I will select features for clustering
feature_cols = ["daily_return", "volatility_pct", "volume", "news_count", "avg_sentiment"]
X = q5_df[feature_cols].values

# Standardizing the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Will use KMeans clustering with (k=3)
k = 3
kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
q5_df["cluster"] = kmeans.fit_predict(X_scaled)

# Converting cluster to string for coloring
q5_df["cluster_str"] = q5_df["cluster"].astype(str)

cluster_names = {
    0: "Calm / Low-News Days",
    1: "High-Volatility Days",
    2: "News-Driven Active Days"
}

q5_df["cluster_label"] = q5_df["cluster"].map(cluster_names)

# Finally will prepare data for Bokeh
source_q5 = ColumnDataSource(q5_df)

# Will use daily_return vs volatility_pct for plotting
p5 = figure(
    width=900,
    height=450,
    title="Clusters of IBM Trading Days (Based on News and Market Features)",
    x_axis_label="Daily Return (%)",
    y_axis_label="Volatility (%)",
    toolbar_location="above"
)

palette = Category10[3]
p5.circle(
    x="daily_return",
    y="volatility_pct",
    source=source_q5,
    size=8,
    alpha=0.8,
    legend_field="cluster_label",
    color=factor_cmap("cluster_label", palette=palette, factors=list(cluster_names.values()))
)

# Adding hover tool for interpretation
hover_q5 = HoverTool(
    tooltips=[
        ("Date", "@date{%F}"),
        ("Cluster", "@cluster_str"),
        ("Return (%)", "@daily_return{0.2f}"),
        ("Volatility (%)", "@volatility_pct{0.2f}"),
        ("Volume", "@volume{0,0}"),
        ("Articles", "@news_count"),
        ("Avg Sentiment", "@avg_sentiment{0.2f}")
    ],
    formatters={"@date": "datetime"}
)
p5.add_tools(hover_q5)

p5.legend.title = "Cluster"
p5.legend.location = "top_left"
p5.legend.click_policy = "hide"

show(p5)

##D4. Summary of key findings

So after running all the analysis, here’s my overall understanding of the data from the two APIs I picked, Alpha Vantage for IBM stock prices and Mediastack for IBM-related news.

1. I plotted the stock price trend. The line chart showed that Open, High, Low, and Close prices move almost together, meaning the stock has a steady, predictable pattern. From August to December, the price slowly climbs upward with normal dips, nothing unusual, just regular market movement.

2. Then I compared news sentiment with stock reaction. Sentiment jumps up and down mostly in October and November, and that’s the same time the stock shows wider High-Low gaps and slightly bigger swings. This tells me the stock becomes a bit more sensitive when news tone is unstable.

3. Next, I checked news volume versus trading activity. The trend line showed that on days when IBM is mentioned more in the news, the trading volume also increases a little. So basically more news, more eyes on the stock, more people trading it.

4. After that, I visualized volatility. Again, October and November stood out, the months with the most uneven news coverage also show a slight rise in volatility. Not too dramatic, but clearly visible.

5. And finally, I used clustering to label market days. The graph split naturally into calm days, high-volatility days, and news-active days. Calm days stay tight and stable, volatility days scatter wider, and news-heavy days lean toward higher activity or fluctuation.

So if I sum it up in one line: IBM’s stock doesn’t crash or spike wildly, but it definitely gets a little more active during months or days when there is more noise and sentiment movement in the news.

##D5 Further research

So for future research, I want to take this project one step ahead, but still keep it simple and meaningful.

1. I plan to bring in stock price data for two other companies: Walmart and Nvidia. Walmart feels like the most stable, steady stock, while Nvidia looks like the opposite — fast growing, almost exponential because of the AI boom. Once I pull that data, I want to compare IBM with both, mainly to see how differently the market reacts to the same kind of media attention when a company is calm versus when it’s rapidly scaling.

2. One more idea could be checking big external events like interest rate announcements or chip embargo news and just seeing if all three stocks react at the same time or if IBM reacts more slowly or differently.
